# 06D — Final Comparative Evaluation + Paired Bootstrap — FINAL V1

Notebook ini dijalankan setelah **06A, 06B, dan 06C COMPLETE**.

Fungsi:
- menggabungkan hasil final Zero-Shot, FSP, dan LoRA;
- membandingkan corpus chrF++, BLEU, dan mean S3E;
- menghitung **paired bootstrap 10.000 resample** pada 53 item test;
- menghasilkan 95% confidence interval untuk selisih skor antar-skenario;
- menghitung probabilitas bootstrap bahwa skenario A lebih baik dari B;
- menyimpan tabel ranking final.

**Arah interpretasi**
- chrF++: lebih tinggi lebih baik (metrik utama)
- BLEU: lebih tinggi lebih baik
- S3E: lebih rendah = keluaran lebih stabil secara semantik

S3E tidak digabungkan secara numerik ke chrF++/BLEU dan tidak dipakai untuk memilih output MBR.

## 1. Environment

In [1]:
from __future__ import annotations

import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

EVAL_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_evaluation"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "final_comparison"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SCENARIOS = {
    "zero_shot": "Zero-Shot",
    "fsp_locked": "Fragment-Shot Prompting",
    "lora_locked": "Fine-Tuning LoRA",
}

N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 42
CI_LOW = 2.5
CI_HIGH = 97.5

bleu = BLEU(
    tokenize="13a",
)
chrfpp = CHRF(
    word_order=2,
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("N_BOOTSTRAP  :", N_BOOTSTRAP)
print("SEED         :", BOOTSTRAP_SEED)

PROJECT_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi
N_BOOTSTRAP  : 10000
SEED         : 42


## 2. Load hasil 06A–06C

In [2]:
def newest_complete_eval(
    scenario_slug: str,
):
    scenario_root = (
        EVAL_ROOT / scenario_slug
    )

    candidates = sorted(
        scenario_root.glob(
            "run_*/final_metric_summary.json"
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for summary_path in candidates:
        try:
            summary = json.loads(
                summary_path.read_text(
                    encoding="utf-8"
                )
            )
            detail_path = (
                summary_path.parent
                / "final_metric_details.csv"
            )
            manifest_path = (
                summary_path.parent
                / "evaluation_manifest.json"
            )

            if not (
                detail_path.exists()
                and manifest_path.exists()
            ):
                continue

            details = pd.read_csv(
                detail_path,
                dtype={"row_id": str},
                keep_default_na=False,
                encoding="utf-8-sig",
            )

            if len(details) != 53:
                continue

            required = {
                "row_id",
                "reference_text",
                "selected_output",
                "s3e_value",
            }
            if not required.issubset(
                details.columns
            ):
                continue

            return (
                summary_path.parent,
                summary,
                details,
            )

        except Exception:
            continue

    raise FileNotFoundError(
        f"Tidak ditemukan output evaluasi COMPLETE untuk {scenario_slug}"
    )


loaded = {}

for slug, label in SCENARIOS.items():
    run_dir, summary, details = (
        newest_complete_eval(slug)
    )
    loaded[slug] = {
        "label": label,
        "run_dir": run_dir,
        "summary": summary,
        "details": details,
    }

    print(
        f"{label:28} -> {run_dir}"
    )

# Cross-scenario alignment audit
base = loaded["zero_shot"]["details"][
    ["row_id", "reference_text"]
].copy()

for slug in [
    "fsp_locked",
    "lora_locked",
]:
    other = loaded[slug]["details"][
        ["row_id", "reference_text"]
    ].copy()

    merged_check = base.merge(
        other,
        on="row_id",
        how="inner",
        validate="one_to_one",
        suffixes=("_a", "_b"),
    )

    if len(merged_check) != 53:
        raise RuntimeError(
            f"Alignment cross-scenario gagal untuk {slug}."
        )

    if not (
        merged_check["reference_text_a"]
        == merged_check["reference_text_b"]
    ).all():
        raise RuntimeError(
            f"Reference berbeda antar skenario pada {slug}."
        )

print("\nCROSS-SCENARIO ALIGNMENT: PASS")

Zero-Shot                    -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/zero_shot/run_1ef71f2034f92d52
Fragment-Shot Prompting      -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065
Fine-Tuning LoRA             -> /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/lora_locked/run_20260813T180837Z

CROSS-SCENARIO ALIGNMENT: PASS


## 3. Tabel perbandingan final

In [3]:
comparison_rows = []

for slug, item in loaded.items():
    s = item["summary"]
    comparison_rows.append({
        "scenario": slug,
        "scenario_label": item["label"],
        "corpus_chrfpp":
            float(s["corpus_chrfpp"]),
        "corpus_bleu_13a":
            float(s["corpus_bleu_13a"]),
        "mean_s3e":
            float(s["mean_s3e"]),
        "median_s3e":
            float(s["median_s3e"]),
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df["rank_chrfpp"] = (
    comparison_df["corpus_chrfpp"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

comparison_df["rank_bleu"] = (
    comparison_df["corpus_bleu_13a"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

comparison_df["rank_s3e_stability"] = (
    comparison_df["mean_s3e"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

comparison_df = comparison_df.sort_values(
    [
        "rank_chrfpp",
        "rank_bleu",
    ],
    kind="stable",
)

display(comparison_df)

,scenario,scenario_label,corpus_chrfpp,corpus_bleu_13a,mean_s3e,median_s3e,rank_chrfpp,rank_bleu,rank_s3e_stability
2,lora_locked,Fine-Tuning LoRA,33.064789,9.191746,0.540777,0.563805,1,2,1
1,fsp_locked,Fragment-Shot Prompting,31.839264,10.565661,0.691243,0.718427,2,1,2
0,zero_shot,Zero-Shot,18.976476,1.295909,0.741564,0.716870,3,3,3


## 4. Paired bootstrap

In [4]:
rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

pairs = [
    ("fsp_locked", "zero_shot"),
    ("lora_locked", "zero_shot"),
    ("lora_locked", "fsp_locked"),
]

bootstrap_rows = []

def corpus_metric(
    metric_name: str,
    hyps,
    refs,
):
    if metric_name == "chrF++":
        return float(
            chrfpp.corpus_score(
                hyps,
                [refs],
            ).score
        )
    if metric_name == "BLEU":
        return float(
            bleu.corpus_score(
                hyps,
                [refs],
            ).score
        )
    raise ValueError(metric_name)


for a_slug, b_slug in pairs:
    a = (
        loaded[a_slug]["details"]
        .sort_values("row_id")
        .reset_index(drop=True)
    )
    b = (
        loaded[b_slug]["details"]
        .sort_values("row_id")
        .reset_index(drop=True)
    )

    if not (
        a["row_id"].tolist()
        == b["row_id"].tolist()
    ):
        raise RuntimeError(
            f"row_id mismatch: {a_slug} vs {b_slug}"
        )

    refs = a[
        "reference_text"
    ].astype(str).tolist()

    if refs != b[
        "reference_text"
    ].astype(str).tolist():
        raise RuntimeError(
            f"reference mismatch: {a_slug} vs {b_slug}"
        )

    a_hyps = a[
        "selected_output"
    ].astype(str).tolist()

    b_hyps = b[
        "selected_output"
    ].astype(str).tolist()

    n = len(refs)

    for metric_name in [
        "chrF++",
        "BLEU",
    ]:
        observed_a = corpus_metric(
            metric_name,
            a_hyps,
            refs,
        )
        observed_b = corpus_metric(
            metric_name,
            b_hyps,
            refs,
        )
        observed_delta = (
            observed_a - observed_b
        )

        deltas = np.empty(
            N_BOOTSTRAP,
            dtype=np.float64,
        )

        for t in range(
            N_BOOTSTRAP
        ):
            idx = rng.integers(
                0,
                n,
                size=n,
            )

            refs_bs = [
                refs[i] for i in idx
            ]
            a_bs = [
                a_hyps[i] for i in idx
            ]
            b_bs = [
                b_hyps[i] for i in idx
            ]

            deltas[t] = (
                corpus_metric(
                    metric_name,
                    a_bs,
                    refs_bs,
                )
                -
                corpus_metric(
                    metric_name,
                    b_bs,
                    refs_bs,
                )
            )

        ci_low, ci_high = (
            np.percentile(
                deltas,
                [CI_LOW, CI_HIGH],
            )
        )

        prob_a_better = float(
            np.mean(
                deltas > 0
            )
        )
        prob_b_better = float(
            np.mean(
                deltas < 0
            )
        )
        prob_tie = float(
            np.mean(
                np.isclose(
                    deltas,
                    0.0,
                    atol=1e-12,
                )
            )
        )

        empirical_two_sided_p = float(
            min(
                1.0,
                2.0
                * min(
                    np.mean(
                        deltas <= 0
                    ),
                    np.mean(
                        deltas >= 0
                    ),
                ),
            )
        )

        bootstrap_rows.append({
            "scenario_a":
                a_slug,
            "scenario_a_label":
                loaded[a_slug]["label"],
            "scenario_b":
                b_slug,
            "scenario_b_label":
                loaded[b_slug]["label"],
            "metric":
                metric_name,
            "score_a":
                observed_a,
            "score_b":
                observed_b,
            "delta_a_minus_b":
                observed_delta,
            "bootstrap_ci95_low":
                float(ci_low),
            "bootstrap_ci95_high":
                float(ci_high),
            "probability_a_better":
                prob_a_better,
            "probability_b_better":
                prob_b_better,
            "probability_tie":
                prob_tie,
            "empirical_two_sided_p":
                empirical_two_sided_p,
            "n_items":
                n,
            "n_bootstrap":
                N_BOOTSTRAP,
            "seed":
                BOOTSTRAP_SEED,
        })

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

display(bootstrap_df)

,scenario_a,scenario_a_label,scenario_b,scenario_b_label,metric,score_a,score_b,delta_a_minus_b,bootstrap_ci95_low,bootstrap_ci95_high,probability_a_better,probability_b_better,probability_tie,empirical_two_sided_p,n_items,n_bootstrap,seed
0,fsp_locked,Fragment-Shot Prompting,zero_shot,Zero-Shot,chrF++,31.839264,18.976476,12.862788,8.048067,18.291086,1.0000,0.0000,0.0,0.0000,53,10000,42
1,fsp_locked,Fragment-Shot Prompting,zero_shot,Zero-Shot,BLEU,10.565661,1.295909,9.269752,3.860810,15.469175,1.0000,0.0000,0.0,0.0000,53,10000,42
2,lora_locked,Fine-Tuning LoRA,zero_shot,Zero-Shot,chrF++,33.064789,18.976476,14.088313,9.889713,19.048390,1.0000,0.0000,0.0,0.0000,53,10000,42
3,lora_locked,Fine-Tuning LoRA,zero_shot,Zero-Shot,BLEU,9.191746,1.295909,7.895837,3.262349,13.100303,1.0000,0.0000,0.0,0.0000,53,10000,42
4,lora_locked,Fine-Tuning LoRA,fsp_locked,Fragment-Shot Prompting,chrF++,33.064789,31.839264,1.225525,-3.227148,5.274652,0.7082,0.2918,0.0,0.5836,53,10000,42
5,lora_locked,Fine-Tuning LoRA,fsp_locked,Fragment-Shot Prompting,BLEU,9.191746,10.565661,-1.373915,-7.554087,4.259188,0.3244,0.6756,0.0,0.6488,53,10000,42


## 5. Simpan hasil

In [5]:
COMPARISON_PATH = (
    OUTPUT_DIR
    / "final_comparison_metrics.csv"
)
BOOTSTRAP_PATH = (
    OUTPUT_DIR
    / "paired_bootstrap_results.csv"
)
SUMMARY_PATH = (
    OUTPUT_DIR
    / "final_comparison_summary.json"
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig",
)

bootstrap_df.to_csv(
    BOOTSTRAP_PATH,
    index=False,
    encoding="utf-8-sig",
)

best_chrf = (
    comparison_df
    .sort_values(
        "corpus_chrfpp",
        ascending=False,
    )
    .iloc[0]
)

best_bleu = (
    comparison_df
    .sort_values(
        "corpus_bleu_13a",
        ascending=False,
    )
    .iloc[0]
)

most_stable = (
    comparison_df
    .sort_values(
        "mean_s3e",
        ascending=True,
    )
    .iloc[0]
)

summary = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "primary_metric":
        "chrF++",
    "best_by_chrfpp": {
        "scenario":
            best_chrf["scenario"],
        "label":
            best_chrf["scenario_label"],
        "score":
            float(
                best_chrf["corpus_chrfpp"]
            ),
    },
    "best_by_bleu": {
        "scenario":
            best_bleu["scenario"],
        "label":
            best_bleu["scenario_label"],
        "score":
            float(
                best_bleu["corpus_bleu_13a"]
            ),
    },
    "most_stable_by_mean_s3e": {
        "scenario":
            most_stable["scenario"],
        "label":
            most_stable["scenario_label"],
        "mean_s3e":
            float(
                most_stable["mean_s3e"]
            ),
    },
    "bootstrap": {
        "method":
            "paired bootstrap resampling over 53 test items",
        "n_bootstrap":
            N_BOOTSTRAP,
        "seed":
            BOOTSTRAP_SEED,
        "ci":
            "percentile 95%",
        "note":
            "chrF++ is primary; BLEU and S3E are supporting evidence.",
    },
    "outputs": {
        "comparison":
            str(COMPARISON_PATH),
        "paired_bootstrap":
            str(BOOTSTRAP_PATH),
    },
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("COMPARISON:", COMPARISON_PATH)
print("BOOTSTRAP :", BOOTSTRAP_PATH)
print("SUMMARY   :", SUMMARY_PATH)

COMPARISON: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_comparison/final_comparison_metrics.csv
BOOTSTRAP : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_comparison/paired_bootstrap_results.csv
SUMMARY   : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_comparison/final_comparison_summary.json


## 6. Final audit

In [6]:
checks = {
    "three_scenarios":
        len(comparison_df) == 3,
    "six_pair_metric_rows":
        len(bootstrap_df) == 6,
    "comparison_exists":
        COMPARISON_PATH.exists(),
    "bootstrap_exists":
        BOOTSTRAP_PATH.exists(),
    "summary_exists":
        SUMMARY_PATH.exists(),
}

failed = [
    k for k, v in checks.items()
    if not bool(v)
]

display(
    pd.DataFrame(
        [
            {"check": k, "ok": bool(v)}
            for k, v in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL COMPARISON AUDIT GAGAL: {failed}"
    )

print("\n" + "=" * 76)
print("FINAL COMPARATIVE EVALUATION + PAIRED BOOTSTRAP: COMPLETE")
print("=" * 76)

display(
    comparison_df[
        [
            "scenario_label",
            "corpus_chrfpp",
            "corpus_bleu_13a",
            "mean_s3e",
            "rank_chrfpp",
            "rank_bleu",
            "rank_s3e_stability",
        ]
    ]
)

print(
    "\nTahap berikutnya setelah ini: "
    "LLM evaluator / analisis kesalahan pada output MBR terpilih."
)

,check,ok
0,three_scenarios,True
1,six_pair_metric_rows,True
2,comparison_exists,True
3,bootstrap_exists,True
4,summary_exists,True



FINAL COMPARATIVE EVALUATION + PAIRED BOOTSTRAP: COMPLETE


,scenario_label,corpus_chrfpp,corpus_bleu_13a,mean_s3e,rank_chrfpp,rank_bleu,rank_s3e_stability
2,Fine-Tuning LoRA,33.064789,9.191746,0.540777,1,2,1
1,Fragment-Shot Prompting,31.839264,10.565661,0.691243,2,1,2
0,Zero-Shot,18.976476,1.295909,0.741564,3,3,3



Tahap berikutnya setelah ini: LLM evaluator / analisis kesalahan pada output MBR terpilih.
